'''
This notebook is intended to create the dataset required for clustering resistance project. 
There are three features: frequency, g-score and proximity to nearest non-self R mutation.
1. Frequency is derived from WHO catalogue: (Present_R)/(Present_R+Present_S). These are column names from the WHO catalogue, each column has a integer number denoting how many times each residue was found
in a resistant isolate or a susceptible isolate. Frequency is denoting the number of times a residue is found in resistant isolates compared to total number of appearances.
2. G-score: add the g-score precomputed to this dataset. Adjust for rpob gene.
3. proximity score: we use the 3D distance maps to calculate the proximity score to the nearest non-self R residue. In case of rpoB, we need to adjust the offset between PDB and WHO indices.
'''

In [1]:
import pandas as pd
import numpy as np
import re
import csv
import os
from evcouplings.compare import DistanceMap

## generate 3-to-1 letter AA codes

In [2]:
# check if the genes of interest are in catalog df
protein_sequences_file = '../data/catalog/protein_sequences.csv'
protein_sequences_df = pd.read_csv(protein_sequences_file)
# Extract genes of interest
genes_of_interest = protein_sequences_df['gene'].unique()

In [7]:
who_catalog_path = '../data/catalog/sanjana_table3.csv'  # Update the path as necessary
who_catalog = pd.read_csv(who_catalog_path)
# Step 3: Check the flattened columns
print(who_catalog.columns)

# Step 1: Select the columns with the correct names
catalog_df = who_catalog[[
    'Drug',
    'gene',
    'mutation', 
    'WHO_Present_R', 
    'WHO_Present_S', 
    'REGRESSION + GRADING RULES'
]]

# Step 2: Rename the columns to more meaningful names (optional)
catalog_df.columns = ['drug','gene', 'variant', 'Present_R', 'Present_S', 'FINAL_CONFIDENCE_GRADING']

# Step 3: Check the first few rows of the selected data
print(catalog_df.head())
# Extract relevant columns

# Filter out rows with NaN values in the 'drug' or 'variant' columns
catalog_df = catalog_df.dropna(subset=['drug', 'variant'])

# subset for target genes
filtered_df = catalog_df[catalog_df['variant'].str.startswith(tuple(genes_of_interest))]


Index(['Drug', 'gene', 'mutation', 'predicted_effect', 'WHO_Odds_Ratio',
       'WHO_pval', 'WHO_BH_pval', 'WHO_neutral_pval', 'WHO_BH_neutral_pval',
       'WHO_LRT_pval', 'WHO_BH_LRT_pval', 'WHO_Present_SR', 'WHO_Present_R',
       'WHO_Present_S', 'WHO_Absent_S', 'WHO_Absent_R', 'WHO_R_PPV',
       'WHO_S_PPV', 'WHO_NPV', 'WHO_Sens', 'WHO_Spec', 'WHO_R_PPV_LB',
       'WHO_R_PPV_UB', 'WHO_S_PPV_LB', 'WHO_S_PPV_UB', 'WHO_NPV_LB',
       'WHO_NPV_UB', 'WHO_Sens_LB', 'WHO_Sens_UB', 'WHO_Spec_LB',
       'WHO_Spec_UB', 'Initial confidence grading WHO dataset',
       'ALL_Odds_Ratio', 'ALL_pval', 'ALL_BH_pval', 'ALL_neutral_pval',
       'ALL_BH_neutral_pval', 'ALL_LRT_pval', 'ALL_BH_LRT_pval',
       'ALL_Present_SR', 'ALL_Present_R', 'ALL_Present_S', 'ALL_Absent_S',
       'ALL_Absent_R', 'ALL_R_PPV', 'ALL_S_PPV', 'ALL_NPV', 'ALL_Sens',
       'ALL_Spec', 'ALL_R_PPV_LB', 'ALL_R_PPV_UB', 'ALL_S_PPV_LB',
       'ALL_S_PPV_UB', 'ALL_NPV_LB', 'ALL_NPV_UB', 'ALL_Sens_LB',
       'ALL_Sens_

In [8]:
# Display the filtered DataFrame
print("number of isolates before discarding invalid residues", len(filtered_df))
# Discard rows that have 'ins' or 'del' in the variant column
filtered_df = filtered_df[~filtered_df['variant'].str.contains('ins|del')]
filtered_df=filtered_df.rename(columns={"FINAL_CONFIDENCE_GRADING": "confidence"})

number of isolates before discarding invalid residues 11159


In [9]:
# Mapping from three-letter to one-letter amino acid codes
three_to_one = {
    'Ala': 'A', 'Cys': 'C', 'Asp': 'D', 'Glu': 'E', 'Phe': 'F', 'Gly': 'G', 'His': 'H',
    'Ile': 'I', 'Lys': 'K', 'Leu': 'L', 'Met': 'M', 'Asn': 'N', 'Pro': 'P', 'Gln': 'Q',
    'Arg': 'R', 'Ser': 'S', 'Thr': 'T', 'Val': 'V', 'Trp': 'W', 'Tyr': 'Y'
}

def convert_three_to_one(mutation):
    # Modify the regex to match the gene name but ignore it in the output
    match = re.match(r"[a-zA-Z0-9]+_p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})", mutation)
    if not match:
        return None
    original_aa, position, new_aa = match.groups()
    try:
        # Return just the one-letter mutation string
        return f"p.{three_to_one[original_aa]}{position}{three_to_one[new_aa]}"
    except KeyError:
        return None

# Example usage:
print(convert_three_to_one("atpE_p.Asp28Gly"))  # Output: p.D28G
print(convert_three_to_one("atpE_p.Ala249Thr"))  # Output: p.A249T


p.D28G
p.A249T


In [10]:
## apply the 3-to-1 conversion
filtered_df['one_letter_mutation'] = filtered_df['variant'].apply(convert_three_to_one)

In [11]:
filtered_df = filtered_df.dropna(subset=['one_letter_mutation'])
filtered_df =filtered_df.drop_duplicates()

In [12]:
filtered_df.head()

,drug,gene,variant,Present_R,Present_S,confidence,one_letter_mutation
1093,Bedaquiline,Rv0678,Rv0678_p.Met146Thr,11.0,0.0,3) Uncertain significance,p.M146T
1094,Bedaquiline,atpE,atpE_p.Ala63Pro,4.0,0.0,2) Assoc w R - Interim,p.A63P
1097,Bedaquiline,atpE,atpE_p.Ile66Met,7.0,0.0,1) Assoc w R,p.I66M
1098,Bedaquiline,Rv0678,Rv0678_p.Ile67Ser,11.0,0.0,2) Assoc w R - Interim,p.I67S
1103,Bedaquiline,Rv0678,Rv0678_p.Ala36Val,4.0,0.0,2) Assoc w R - Interim,p.A36V


In [13]:
#check for unique genes
print(np.unique(filtered_df['gene']))

['Rv0678' 'atpE' 'ddn' 'embB' 'ethA' 'fgd1' 'gid' 'gyrA' 'gyrB' 'inhA'
 'katG' 'pepQ' 'pncA' 'rplC' 'rpoB' 'rpsL' 'tlyA']


In [14]:
filtered_df.shape

(6116, 7)

In [15]:
# Desired column order
first_cols = ['drug', 'gene', 'variant']

# Get all other columns that are not in first_cols
remaining_cols = [col for col in filtered_df.columns if col not in first_cols]

# Reorder the DataFrame
filtered_df = filtered_df[first_cols + remaining_cols]


In [16]:
filtered_df.to_csv('../data/new_data/mutations_with_one_letter_all_confidence.csv', index=False)

## feature 1: frequency

In [17]:
# Calculate the frequency of each variant in the population
filtered_df['frequency'] = (filtered_df['Present_R']) / (filtered_df['Present_R'] + filtered_df['Present_S'])

In [18]:
filtered_df.head(5)

,drug,gene,variant,Present_R,Present_S,confidence,one_letter_mutation,frequency
1093,Bedaquiline,Rv0678,Rv0678_p.Met146Thr,11.0,0.0,3) Uncertain significance,p.M146T,1.0
1094,Bedaquiline,atpE,atpE_p.Ala63Pro,4.0,0.0,2) Assoc w R - Interim,p.A63P,1.0
1097,Bedaquiline,atpE,atpE_p.Ile66Met,7.0,0.0,1) Assoc w R,p.I66M,1.0
1098,Bedaquiline,Rv0678,Rv0678_p.Ile67Ser,11.0,0.0,2) Assoc w R - Interim,p.I67S,1.0
1103,Bedaquiline,Rv0678,Rv0678_p.Ala36Val,4.0,0.0,2) Assoc w R - Interim,p.A36V,1.0


In [19]:
filtered_df.drop(columns=['Present_R','Present_S'], inplace=True)

In [20]:
filtered_df.head()

,drug,gene,variant,confidence,one_letter_mutation,frequency
1093,Bedaquiline,Rv0678,Rv0678_p.Met146Thr,3) Uncertain significance,p.M146T,1.0
1094,Bedaquiline,atpE,atpE_p.Ala63Pro,2) Assoc w R - Interim,p.A63P,1.0
1097,Bedaquiline,atpE,atpE_p.Ile66Met,1) Assoc w R,p.I66M,1.0
1098,Bedaquiline,Rv0678,Rv0678_p.Ile67Ser,2) Assoc w R - Interim,p.I67S,1.0
1103,Bedaquiline,Rv0678,Rv0678_p.Ala36Val,2) Assoc w R - Interim,p.A36V,1.0


In [21]:
filtered_df.to_csv('../data/new_data/all_proteins_frequency_catalog.csv', index=False)

## feature 2: proximity to nearest r-conferring mutations

### helper codes

In [22]:
def extract_position_from_mutation(mutation):
    ## to extract position from mutation column
    match = re.search(r'\d+', mutation)
    if match:
        return int(match.group(0))
    return None
# def adjust_number(index):
#     # adjust index offset between pdb and who indices for rpob
#     if index < 30:  # Assuming offset starts at position 23 + 7
#         return index + 7
#     elif 30 <= index <= 1180:  # Adjusts up to the maximum offset point
#         return index + 6
#     else:
#         return index

def adjust_number(index):
    # adjust index offset between pdb and who indices for rpob
    return index + 6 


def proximity_1D(gene_subset, unique_valid_r_positions, gene_name):
    proximity_to_nonself_r_conferring = []
    positions_causing_error = []
    nearest_mutation_index = []
    for index, row in gene_subset.iterrows():
        #take the WHO adjusted position wrt PDB for rpoB and position for all 
        if gene_name=='rpoB':
            current_position = row['WHO_Adjusted_Position']
        else:
            current_position= row['position']
        distances = []

        for r_pos in unique_valid_r_positions:
            # iterate through list of R positions we have to compute against the current position
            if r_pos != current_position: #if its a non-self R residue (which we expect)

                try:
                    dist = abs(current_position - r_pos)
                    if not np.isnan(dist):
                        # if distance is not NaN, add it to the dist dict with the current R residue position
                        distances.append((dist, r_pos))
                except KeyError:
                    #keep a log of positions causing Nan dist value despite taking the diff between self and nonself R positions
                    positions_causing_error.append((current_position, r_pos))
                    continue

        if distances:
            #take the min of all distances for per residue
            min_distance, min_index = min(distances, key=lambda x: x[0])
        else:
            min_distance, min_index = np.nan, np.nan
        # Append results to the lists so that we can just add this list as a column to the dataframe
        proximity_to_nonself_r_conferring.append(min_distance)
        nearest_mutation_index.append(min_index)
    # Store the results in the phenotype_data DataFrame
    gene_subset['Proximity_1D'] = proximity_to_nonself_r_conferring
    gene_subset['Nearest_1D_Index'] = nearest_mutation_index
    return gene_subset    

def non_self_proximity_r_mutants(gene_subset, unique_valid_r_positions, distance_map, gene_name):
    positions_causing_error = []
    proximity_to_nonself_r_conferring = []
    nearest_mutation_index = []

    for index, row in gene_subset.iterrows():
        #take the WHO adjusted position wrt PDB for rpoB and position for all 
        if gene_name=='rpoB':
            current_position = row['WHO_Adjusted_Position']
        else:
            current_position= row['position']
        distances = []

        for r_pos in unique_valid_r_positions:
            # iterate through list of R positions we have to compute against the current position
            if r_pos != current_position: #if its a non-self R residue (which we expect)

                try:
                    dist = distance_map.dist(current_position, r_pos, raise_na=True)
                    if not np.isnan(dist):
                        # if distance is not NaN, add it to the dist dict with the current R residue position
                        distances.append((dist, r_pos))
                except KeyError:
                    #keep a log of positions causing Nan dist value despite taking the diff between self and nonself R positions
                    positions_causing_error.append((current_position, r_pos))
                    continue

        if distances:
            #take the min of all distances for per residue
            min_distance, min_index = min(distances, key=lambda x: x[0])
        else:
            min_distance, min_index = np.nan, np.nan
        # Append results to the lists so that we can just add this list as a column to the dataframe
        proximity_to_nonself_r_conferring.append(min_distance)
        nearest_mutation_index.append(min_index)
    # Store the results in the phenotype_data DataFrame
    gene_subset['Proximity_to_R_Conferring'] = proximity_to_nonself_r_conferring
    gene_subset['Nearest_Mutation_Index'] = nearest_mutation_index
    return gene_subset
# Map confidence levels to phenotypes
def map_confidence_to_phenotype(confidence):
    if confidence in ['1) Assoc w R', '2) Assoc w R - Interim']:
        return 'Resistant'
    elif confidence in ['4) Not assoc w R - Interim', '5) Not assoc w R']:
        return 'Susceptible'
    else:
        return 'Unknown'

### compute proximity

In [23]:
#load the derived dataframe from previous step
catalog_df=pd.read_csv('../data/new_data/all_proteins_frequency_catalog.csv')
catalog_df['phenotype'] =catalog_df['confidence'].apply(map_confidence_to_phenotype)
print("distribution of phenotype", catalog_df['phenotype'].value_counts())
# extract position from mutation and create a column
catalog_df['position'] =catalog_df['one_letter_mutation'].str.extract(r'(\d+)').astype(int)
#Get the unique combinations of gene and drug
unique_genes_drugs = catalog_df[['gene', 'drug']].drop_duplicates()

# Print each gene and drug pair - debug step
for index, row in unique_genes_drugs.iterrows():
    print(f"Gene: {row['gene']}, Drug: {row['drug']}")
#create list of unique gene names
gene_names=list(np.unique(catalog_df['gene']))

distribution of phenotype phenotype
Unknown        5495
Resistant       571
Susceptible      50
Name: count, dtype: int64
Gene: Rv0678, Drug: Bedaquiline
Gene: atpE, Drug: Bedaquiline
Gene: pepQ, Drug: Bedaquiline
Gene: tlyA, Drug: Capreomycin
Gene: pepQ, Drug: Clofazimine
Gene: Rv0678, Drug: Clofazimine
Gene: ddn, Drug: Delamanid
Gene: fgd1, Drug: Delamanid
Gene: embB, Drug: Ethambutol
Gene: inhA, Drug: Ethionamide
Gene: ethA, Drug: Ethionamide
Gene: katG, Drug: Isoniazid
Gene: inhA, Drug: Isoniazid
Gene: gyrA, Drug: Levofloxacin
Gene: gyrB, Drug: Levofloxacin
Gene: rplC, Drug: Linezolid
Gene: gyrA, Drug: Moxifloxacin
Gene: gyrB, Drug: Moxifloxacin
Gene: pncA, Drug: Pyrazinamide
Gene: rpoB, Drug: Rifampicin
Gene: rpsL, Drug: Streptomycin
Gene: gid, Drug: Streptomycin


In [24]:
##load the protein details file - we get the dist map id from it
protein_details_path = '../data/catalog/17_proteins_details.xlsx'  # Update the path as necessary
protein_details = pd.read_excel(protein_details_path, sheet_name='Sheet1')
# Filter the DataFrame for the genes of interest
filtered_df = protein_details[protein_details['gene_name'].str.contains('|'.join(gene_names), case=False, na=False)]
print("previewing protein details file")
filtered_df.head()

previewing protein details file


,Drug,gene_name,filename,Uniprot,references,Entry,Protein names,rv,drug_full,Aliases,start_position_on_the_genomic_accession,end_position_on_the_genomic_accession,orientation
0,RIF,rpoB,rpoBC.fasta,RPOB_MYCTU,"WHO 2021 catalog, category 1",P9WGY9,DNA-directed RNA polymerase subunit beta (RNAP...,Rv0667,rifampicin,Rv0667,759806.0,763324.0,plus
1,INH,inhA,FabG1-inhA.fasta,INHA_MYCTU,"WHO 2021 catalog, category 1",P9WGR1,Enoyl-[acyl-carrier-protein] reductase [NADH] ...,Rv1484,isoniazid,Rv1484,1674201.0,1675010.0,plus
2,INH,katG,KatG.fasta,KATG_MYCTU,"WHO 2021 catalog, category 1",P9WIE5,Catalase-peroxidase (CP) (EC 1.11.1.21) (Perox...,Rv1908c,isoniazid,Rv1908c,2153889.0,2156111.0,minus
3,EMB,embB,embCAB.fasta,EMBB_MYCTU,"WHO 2021 catalog, category 1",P9WNL7,Probable arabinosyltransferase B (EC 2.4.2.-),Rv3795,ethambutol,Rv3795,4246513.0,4249809.0,plus
4,PZA,pncA,pncA.fasta,PNCA_MYCTU,"WHO 2021 catalog, category 1",I6XD65,Nicotinamidase/pyrazinamidase (Nicotinamidase)...,Rv2043c,pyrazinamide,Rv2043c,2288681.0,2289241.0,minus


In [27]:
# Initialize an empty list to store all processed DataFrames
all_gene_data = []


for index, row in filtered_df.iterrows():
    # Extract metadata from current row
    fasta_filename = row['filename']
    uniprot = row['Uniprot']
    entry = row['Entry']
    drug = row['drug_full']
    drug_code = row['Drug']
    gene_name = row['gene_name']
    
    print(f"\nProcessing {gene_name} from file: {fasta_filename}")

    # Subset catalog to this gene
    gene_subset = catalog_df[catalog_df['gene'] == gene_name].drop_duplicates()

    # Load distance map
    distmap_path = f"/project/pi_annagreen_umass_edu/mahbuba/all_projects/resistance_forecast/data/distmaps/{uniprot}/{entry}"
    dist_map = DistanceMap.from_file(distmap_path)
    gene_length = dist_map.dist_matrix.shape[0]
    print(f"Distance map length for {gene_name}: {gene_length}")

    # Pull residues
    residues_df = dist_map.residues_i.copy()
    residues_df['coord_id'] = residues_df['coord_id'].astype(int)
    
    # Get WHO mutation info
    gene_subset = gene_subset.copy()
    #apply adjust number method if it is rpob
    if gene_name == 'rpoB':
        gene_subset['WHO_Adjusted_Position'] = gene_subset['position'].apply(adjust_number)
        #set the who adjusted position as r-conferring positions array for proximity calculation
        r_conferring_positions = gene_subset[gene_subset['phenotype'] == 'Resistant']['WHO_Adjusted_Position'].tolist()
    else:
        r_conferring_positions = gene_subset[gene_subset['phenotype'] == 'Resistant']['position'].tolist()


    unique_valid_r_positions = np.unique(r_conferring_positions)
    print("Number of Unique valid R-conferring positions:", len(unique_valid_r_positions))

    #proximity -1d
    gene_subset =  proximity_1D(gene_subset, unique_valid_r_positions, gene_name)
    

    # Proximity analysis-proximity 3d
    gene_subset = non_self_proximity_r_mutants(gene_subset, unique_valid_r_positions, dist_map, gene_name)


    # load and merge G-scores
    try:
        g_scores = pd.read_csv(f"/project/pi_annagreen_umass_edu/mahbuba/all_projects/resistance_forecast/data/distmaps/{uniprot}/G_scores.csv")
        print(f"G_scores loaded for {uniprot}")
        if gene_name == 'rpoB':
            # gene_subset['adjusted_position'] = gene_subset['position'].apply(adjust_number)
            merged = pd.merge(gene_subset, g_scores, left_on='WHO_Adjusted_Position', right_on='residue', how='left')
        else:
            #since for other genes we are not adjusting their position, we use the actual column
            merged = pd.merge(gene_subset, g_scores, left_on='position', right_on='residue', how='left')

        merged.drop(columns=['residue', 'Unnamed: 0'], errors='ignore', inplace=True)

        all_gene_data.append(merged)

    except FileNotFoundError as e:
        print(f"Skipping G-score merge for {uniprot}: {e}")
        all_gene_data.append(gene_subset)



Processing rpoB from file: rpoBC.fasta
Distance map length for rpoB: 1138
Number of Unique valid R-conferring positions: 28
G_scores loaded for RPOB_MYCTU

Processing inhA from file: FabG1-inhA.fasta
Distance map length for inhA: 246
Number of Unique valid R-conferring positions: 4
G_scores loaded for INHA_MYCTU

Processing katG from file: KatG.fasta
Distance map length for katG: 716
Number of Unique valid R-conferring positions: 22
G_scores loaded for KATG_MYCTU

Processing embB from file: embCAB.fasta
Distance map length for embB: 1054
Number of Unique valid R-conferring positions: 23
G_scores loaded for EMBB_MYCTU

Processing pncA from file: pncA.fasta
Distance map length for pncA: 185
Number of Unique valid R-conferring positions: 96
G_scores loaded for PNCA_MYCTU

Processing gyrA from file: gyrBA.fasta
Distance map length for gyrA: 766
Number of Unique valid R-conferring positions: 9
G_scores loaded for GYRA_MYCTU

Processing rpsL from file: rpsL.fasta
Distance map length for rps

In [28]:
# After the loop, concatenate all the DataFrames in the list into a single large DataFrame
final_df = pd.concat(all_gene_data, ignore_index=True)

In [29]:
final_df

,drug,gene,variant,confidence,one_letter_mutation,frequency,phenotype,position,WHO_Adjusted_Position,Proximity_1D,Nearest_1D_Index,Proximity_to_R_Conferring,Nearest_Mutation_Index,G_score
0,Rifampicin,rpoB,rpoB_p.Ser450Leu,1) Assoc w R,p.S450L,0.987181,Resistant,450,456.0,1.0,455.0,1.328860,455.0,2.808807
1,Rifampicin,rpoB,rpoB_p.Asp435Val,1) Assoc w R,p.D435V,0.986975,Resistant,435,441.0,1.0,440.0,1.324017,440.0,4.291423
2,Rifampicin,rpoB,rpoB_p.His445Asp,1) Assoc w R,p.H445D,0.987013,Resistant,445,451.0,1.0,450.0,1.329035,452.0,16.133619
3,Rifampicin,rpoB,rpoB_p.His445Tyr,1) Assoc w R,p.H445Y,0.978070,Resistant,445,451.0,1.0,450.0,1.329035,452.0,16.133619
4,Rifampicin,rpoB,rpoB_p.Ser450Trp,1) Assoc w R,p.S450W,0.979695,Resistant,450,456.0,1.0,455.0,1.328860,455.0,2.808807
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6111,Delamanid,fgd1,fgd1_p.Thr255Ala,3) Uncertain significance,p.T255A,NaN,Unknown,255,NaN,NaN,NaN,NaN,NaN,NaN
6112,Delamanid,fgd1,fgd1_p.Lys296Arg,3) Uncertain significance,p.K296R,NaN,Unknown,296,NaN,NaN,NaN,NaN,NaN,NaN
6113,Delamanid,fgd1,fgd1_p.Arg64Ser,3) Uncertain significance,p.R64S,0.000000,Unknown,64,NaN,NaN,NaN,NaN,NaN,NaN
6114,Delamanid,fgd1,fgd1_p.Met93Thr,3) Uncertain significance,p.M93T,NaN,Unknown,93,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
np.unique(final_df['confidence'])

array(['1) Assoc w R', '2) Assoc w R - Interim',
       '3) Uncertain significance', '4) Not assoc w R - Interim',
       '5) Not assoc w R'], dtype=object)

In [31]:
final_df.to_csv("../data/new_data/all_proteins_freq_proximity_gscore.csv",index=False)

## debug code from ANNA

In [27]:
# gene='rpoB'
# rpoB_variants= final_df.query("gene ==@gene").sort_values(by="position", ascending=True)
# rpoB_variants.to_csv("../data/rpoB_all_variants.csv", index=False)
# ##prepare the test case file
# test_case = final_df.query("gene == 'rpoB' and position in [445, 450, 451]")
# test_case.to_csv("all_proteins_freq_proximity_gscore_test_case.csv", index=False)

In [28]:
test_case=pd.read_csv("all_proteins_freq_proximity_gscore_test_case.csv")

In [29]:
## test cases

from Bio import SeqIO
from io import StringIO
import pandas as pd
import requests

def fetch_uniprot_sequence(uniprot_id):
    """
    Fetches a protein sequence from UniProt for a single accession ID.

    Parameters:
        uniprot_id (str): UniProt accession ID.

    Returns:
        SeqRecord: A Biopython SeqRecord object.
    """
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    response = requests.get(url)
    response.raise_for_status()

    fasta_io = StringIO(response.text)
    record = next(SeqIO.parse(fasta_io, "fasta")) 

    return record


    
def verify_mutations_vs_uniprot(subset, entry, check_column):
    """
    checks for sequence match between WHO table and uniprot entry sequence

    subset: pd.DataFrame of WHO table
        position - column that contains the 1 indexed position
        original_AA - column with 1 letter AA code of WT AA

    entry: str
        uniprot entry
    """
    # fetch the sequence from uniprot
    record = fetch_uniprot_sequence(entry)
    print(entry, record.seq)
    mismatches=[]

    for idx,row in subset.iterrows():
        #position was showing float output
        position=int(row[check_column])
        WHO_aa=row['original_AA']

        #check uniprot sequence
        if record.seq[position -1] == WHO_aa:
            continue
        else:
            print(f"Error for protein {record.id}, WHO and Uniprot do not match for position {row[check_column]}. WHO AA: {row.original_AA}, Uniprot_AA:{record.seq[row.position -1]}")
            mismatches.append({
                "UniProt_ID": record.id,
                "Gene": row['gene'],
                "Position": int(row[check_column]),
                "WHO_AA": WHO_aa,
                "UniProt_AA": record.seq[position -1],
                "Phenotype": row['phenotype']
            })
    return mismatches


def run_test():
    catalog_df = pd.read_csv("../data/derived_features/all_proteins_freq_proximity_gscore.csv")

    catalog_df['raw_position'] =catalog_df['one_letter_mutation'].str.extract(r'(\d+)').astype(int)
    catalog_df['original_AA'] =catalog_df['one_letter_mutation'].str.extract(r'(\S{1})\d+')
    catalog_df['mutated_AA'] =catalog_df['one_letter_mutation'].str.extract(r'\d+(\S{1})')
    all_mismatches = []

    ##load the protein details file
    protein_details_path = '../data/catalog/17_proteins_details.xlsx' 
    protein_details = pd.read_excel(protein_details_path, sheet_name='Sheet1')

    output_df_list = []

    # Iterate through each protein
    for idx, row in protein_details.iterrows():
        entry = row.Entry
        gene = row.gene_name
        subset = catalog_df.query("gene==@gene")
        print(f"running debugging for gene: {gene}")
        if gene=='rpoB':
           mismatches = verify_mutations_vs_uniprot(subset, entry, check_column="WHO_Adjusted_Position")
        else:
           mismatches = verify_mutations_vs_uniprot(subset, entry, check_column="position")
        all_mismatches.extend(mismatches)
    # Save to CSV
    mismatch_df = pd.DataFrame(all_mismatches)
    mismatch_df.to_csv("../data/who_uniprot_mismatch_report.csv", index=False)
    print("Saved mismatch report to 'results/who_uniprot_mismatch_report.csv'")

def run_test_on_provided_cases():
    test_df = pd.read_csv("all_proteins_freq_proximity_gscore_test_case.csv")

    test_df['raw_position'] =test_df['one_letter_mutation'].str.extract(r'(\d+)').astype(int)
    test_df['original_AA'] =test_df['one_letter_mutation'].str.extract(r'(\S{1})\d+')
    test_df['mutated_AA'] =test_df['one_letter_mutation'].str.extract(r'\d+(\S{1})')
    all_mismatches = []
    ##load the protein details file
    protein_details_path = '../data/catalog/17_proteins_details.xlsx' 
    protein_details = pd.read_excel(protein_details_path, sheet_name='Sheet1')

    # Iterate through each protein
    for idx, row in protein_details.iterrows():
        entry = row.Entry
        gene = row.gene_name
        subset = test_df.query("gene==@gene")
        print(f"running debugging for gene: {gene}")
        if gene=='rpoB':
           mismatches = verify_mutations_vs_uniprot(subset, entry, check_column="WHO_Adjusted_Position")
        else:
           mismatches = verify_mutations_vs_uniprot(subset, entry, check_column="position")
        all_mismatches.extend(mismatches)


    mismatch_df = pd.DataFrame(all_mismatches)
    mismatch_df.to_csv("test_case_mismatch_output.csv", index=False)
    print("Saved test case mismatch output to 'test_case_mismatch_output.csv'")


In [30]:
run_test()
# run_test_on_provided_cases()

running debugging for gene: rpoB
P9WGY9 MLEGCILADSRQSKTAASPSPSRPQSSSNNSVPGAPNRVSFAKLREPLEVPGLLDVQTDSFEWLIGSPRWRESAAERGDVNPVGGLEEVLYELSPIEDFSGSMSLSFSDPRFDDVKAPVDECKDKDMTYAAPLFVTAEFINNNTGEIKSQTVFMGDFPMMTEKGTFIINGTERVVVSQLVRSPGVYFDETIDKSTDKTLHSVKVIPSRGAWLEFDVDKRDTVGVRIDRKRRQPVTVLLKALGWTSEQIVERFGFSEIMRSTLEKDNTVGTDEALLDIYRKLRPGEPPTKESAQTLLENLFFKEKRYDLARVGRYKVNKKLGLHVGEPITSSTLTEEDVVATIEYLVRLHEGQTTMTVPGGVEVPVETDDIDHFGNRRLRTVGELIQNQIRVGMSRMERVVRERMTTQDVEAITPQTLINIRPVVAAIKEFFGTSQLSQFMDQNNPLSGLTHKRRLSALGPGGLSRERAGLEVRDVHPSHYGRMCPIETPEGPNIGLIGSLSVYARVNPFGFIETPYRKVVDGVVSDEIVYLTADEEDRHVVAQANSPIDADGRFVEPRVLVRRKAGEVEYVPSSEVDYMDVSPRQMVSVATAMIPFLEHDDANRALMGANMQRQAVPLVRSEAPLVGTGMELRAAIDAGDVVVAEESGVIEEVSADYITVMHDNGTRRTYRMRKFARSNHGTCANQCPIVDAGDRVEAGQVIADGPCTDDGEMALGKNLLVAIMPWEGHNYEDAIILSNRLVEEDVLTSIHIEEHEIDARDTKLGAEEITRDIPNISDEVLADLDERGIVRIGAEVRDGDILVGKVTPKGETELTPEERLLRAIFGEKAREVRDTSLKVPHGESGKVIGIRVFSREDEDELPAGVNELVRVYVAQKRKISDGDKLAGRHGNKGVIGKILPVEDMPFLADGTPVDIILNTHGVPRRMNIGQILETHLGWCAHSGWKVDAAKGVPDWAARLP

In [26]:
# Load mismatch report and test case
mismatch_df = pd.read_csv("../data/who_uniprot_mismatch_report.csv")
test_cases = pd.read_csv("all_proteins_freq_proximity_gscore_test_case.csv")

# Merge or filter mismatches that match the test case variants
check_variants = test_cases['one_letter_mutation'].unique()
mismatched_variants = mismatch_df[mismatch_df['Position'].isin(test_cases['position']) &
                                  mismatch_df['Gene'].isin(test_cases['gene'])]


if mismatched_variants.empty:
    print(" All 4 test cases pass the WHO–UniProt sequence validation.")
else:
    print("Found mismatches in test cases:")
    print(mismatched_variants)


 All 4 test cases pass the WHO–UniProt sequence validation.
